# Car Price Valuation — EDA & Multi-Model Benchmarking
Comprehensive Data Preprocessing, Feature Engineering, Outlier Treatment, and Regression Benchmarks (Linear, Ridge, Lasso, Random Forest).

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

%matplotlib inline
sns.set_theme(style="darkgrid")

## 1. Data Ingestion & Schema Inspection

In [3]:
df = pd.read_csv("../data/car_data.csv")
print(f"Shape: {df.shape}")
df.head()

Shape: (157, 16)


,Manufacturer,Model,Sales_in_thousands,__year_resale_value,Vehicle_type,Price_in_thousands,Engine_size,Horsepower,Wheelbase,Width,Length,Curb_weight,Fuel_capacity,Fuel_efficiency,Latest_Launch,Power_perf_factor
0,Acura,Integra,16.919,16.360,Passenger,21.50,1.8,140.0,101.2,67.3,172.4,2.639,13.2,28.0,2/2/2012,58.280150
1,Acura,TL,39.384,19.875,Passenger,28.40,3.2,225.0,108.1,70.3,192.9,3.517,17.2,25.0,6/3/2011,91.370778
2,Acura,CL,14.114,18.225,Passenger,NaN,3.2,225.0,106.9,70.6,192.0,3.470,17.2,26.0,1/4/2012,NaN
3,Acura,RL,8.588,29.725,Passenger,42.00,3.5,210.0,114.6,71.4,196.6,3.850,18.0,22.0,3/10/2011,91.389779
4,Audi,A4,20.397,22.255,Passenger,23.99,1.8,150.0,102.6,68.2,178.0,2.998,16.4,27.0,10/8/2011,62.777639


## 2. Preprocessing & Outlier Handling

In [4]:
key_cols = ["Manufacturer", "Vehicle_type", "Price_in_thousands", "Engine_size", "Fuel_efficiency"]
clean_df = df.dropna(subset=key_cols).copy()
clean_df["Price"] = clean_df["Price_in_thousands"] * 85000
clean_df["Mileage"] = clean_df["Fuel_efficiency"] * 10
clean_df["EngineV"] = clean_df["Engine_size"]
clean_df = clean_df[(clean_df["Price"] > 100000) & (clean_df["EngineV"] <= 6.5)]
clean_df.describe()

,Sales_in_thousands,__year_resale_value,Price_in_thousands,Engine_size,Horsepower,Wheelbase,Width,Length,Curb_weight,Fuel_capacity,Fuel_efficiency,Power_perf_factor,Price,Mileage,EngineV
count,152.000000,117.000000,152.000000,152.000000,152.000000,152.000000,152.000000,152.000000,151.000000,152.000000,152.000000,152.000000,1.520000e+02,152.000000,152.000000
mean,53.457934,17.763419,27.165704,3.026974,183.657895,107.519079,71.084211,187.219079,3.376192,17.955921,23.881579,76.227028,2.309085e+06,238.815789,3.026974
std,68.873420,11.010449,14.068214,0.977358,53.062890,7.672862,3.458957,13.491669,0.638711,3.936915,4.259674,23.698912,1.195798e+06,42.596742,0.977358
min,0.110000,5.160000,9.235000,1.000000,55.000000,92.600000,62.600000,149.400000,1.895000,10.300000,15.000000,23.276272,7.849750e+05,150.000000,1.000000
25%,14.212750,11.240000,17.888750,2.300000,147.500000,103.000000,68.375000,177.575000,2.962500,15.775000,21.000000,59.755537,1.520544e+06,210.000000,2.300000
50%,29.213000,14.010000,22.747000,3.000000,175.000000,107.000000,70.400000,187.250000,3.332000,17.200000,24.000000,71.514623,1.933495e+06,240.000000,3.000000
75%,68.069750,19.875000,31.938750,3.575000,211.250000,112.200000,73.100000,196.125000,3.822000,19.800000,26.000000,89.408406,2.714794e+06,260.000000,3.575000
max,540.561000,67.550000,85.500000,5.700000,345.000000,138.700000,79.900000,224.500000,5.572000,32.000000,45.000000,141.141150,7.267500e+06,450.000000,5.700000


## 3. Comparative Model Evaluation

In [5]:
features = ["EngineV", "Mileage"]
X = clean_df[features]
y = np.log1p(clean_df["Price"])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.01),
    "Random Forest (Best)": RandomForestRegressor(n_estimators=100, random_state=42)
}

res = []
for name, m in models.items():
    m.fit(X_tr, y_tr)
    p = m.predict(X_te)
    res.append({"Model": name, "R2 Score": round(r2_score(y_te, p), 4), "MAE": round(mean_absolute_error(np.expm1(y_te), np.expm1(p)), 2)})
pd.DataFrame(res).sort_values(by="R2 Score", ascending=False)

,Model,R2 Score,MAE
3,Random Forest (Best),0.5558,489091.94
2,Lasso Regression,0.4134,576146.68
1,Ridge Regression,0.4064,578841.79
0,Linear Regression,0.4044,579739.69
